## Day 2

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## Chat completions API

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('cd_api_key')

if not api_key:
    print('API key not found, please check')
else:
    print('API key Found, and looks good so far!')


### Endpoint - Hitting open AI using requests module

In [ ]:
import requests
from App.config import URL, headers, payload

import requests

response = requests.post(
    url=URL,
    headers=headers,
    json=payload
)

response.json()

In [ ]:
response.json()['choices'][0]['message']['content']

### Running using Ollama

In [ ]:
requests.get('http://127.0.0.1:11434/').content

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
from openai import OpenAI
from week1.scraper import fetch_website_contents
from IPython.display import Markdown, display, update_display

### Ollama base url

In [ ]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [ ]:
messages = [
    {'role': 'system', 'content': 'You are an helpful assistant'},
    {'role': 'user', 'content': 'Tell me a fun fact'}
]

response = ollama.chat.completions.create(messages=messages, model='llama3.2:1b')

response.choices[0].message.content


# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

In [ ]:
ed_site = "https://edwarddonner.com"

ed_response = fetch_website_contents(ed_site)
print(ed_response)

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

In [ ]:
def summarize(website_url):
    website = fetch_website_contents(website_url)
    messages = [
        {'role' : 'system', 'content': system_prompt},
        {'role' : 'user', 'content': user_prompt_prefix + '\n\n' + website}
    ]

    streaming = ollama.chat.completions.create(messages=messages, model='llama3.2:1b', stream=True)

    response = ''
    handle_display = display(Markdown(''), display_id=True)

    for chunk in streaming:
        if not chunk.choices:
            continue
        
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=handle_display.display_id)


In [ ]:
summarize(ed_site)